In [14]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer

repo_id = 'microsoft/Phi-3-mini-4k-instruct' # Save model's address from hugging face in a variable
tokenizer = AutoTokenizer.from_pretrained(repo_id) # Picks up the tokenizer from the model itself, it is usually already sbundled with the model
vocab_size = len(tokenizer)


In [5]:
vocab_size

32011

In [7]:
torch.manual_seed(13) # this just makes the experiments reproducable
d_model = 1024 # deciding the embedding dimension
embedding_layer = nn.Embedding(vocab_size, d_model) # Make the embedding layer (lookup the table and convert each token into embeddings or vectors which understands the meaning)
linear_query = nn.Linear(d_model, d_model) # linear projection of query vectors
linear_key = nn.Linear(d_model, d_model) # linear projection of key vectors
linear_value = nn.Linear(d_model, d_model) # linear projection of value vectors

In [12]:
sentence = 'Just a dummy sentence'
input_ids = tokenizer(sentence, return_tensors='pt')['input_ids']
input_ids

tensor([[ 3387,   263, 20254, 10541]])

In [13]:
embeddings = embedding_layer(input_ids)
embeddings.shape

torch.Size([1, 4, 1024])

In [16]:
# Creating projections

proj_key = linear_key(embeddings)
proj_query = linear_query(embeddings)
proj_value = linear_value(embeddings)

# Attention score using dot product
dot_products = torch.matmul(proj_query, proj_key.transpose(-2,-1))
scores = F.softmax(dot_products / np.sqrt(d_model), dim=-1)
scores.shape

torch.Size([1, 4, 4])

In [17]:
context = torch.matmul(scores, proj_value)
context.shape

torch.Size([1, 4, 1024])